# Respondedor de preguntas sobre literatura científica reciente
## IA generativa y uso responsable en la investigación educativa

Este cuaderno responde preguntas usando **solo** los títulos y resúmenes exportados desde Web of Science. El flujo es:

1. Cargar las exportaciones de WoS (una o dos búsquedas) y eliminar duplicados.
2. Convertir cada artículo en un vector (*embedding*) para buscar por significado.
3. Para cada pregunta, recuperar los artículos más relacionados.
4. Pedir a un LLM una respuesta en español, con citas numeradas que el propio código convierte en referencias reales.
5. Verificar que el modelo no cite documentos inexistentes y exportar todo a Excel y Word para la **validación humana**.

> Solo se procesan metadatos públicos (títulos y resúmenes). No se envían datos personales ni sensibles a la API.

# 1. Configuración inicial

**Clave de la API.** No escribas la clave en el código. Guárdala en los Secretos de Colab:
panel izquierdo → ícono de llave 🔑 → *Add new secret* → nombre `OPENAI_API_KEY` → pega tu clave → activa *Notebook access*.

In [ ]:
#@title <b><font color="blue" size="+2"> API key de OpenAI (desde Secretos de Colab)
from google.colab import userdata
clave_api = userdata.get("OPENAI_API_KEY")
print("Clave cargada correctamente." if clave_api else "No se encontró la clave OPENAI_API_KEY en Secretos.")

Clave cargada correctamente.


In [ ]:
!pip install -q openai python-docx pandas xlrd openpyxl

from openai import OpenAI
import pandas as pd
import numpy as np
import os, re, time
from datetime import date
from docx import Document

In [ ]:
#@title <b><font color="blue" size="+2"> Modelos y parámetros
modelo_respuesta = "gpt-6-luna"  #@param {type:"string"}
modelo_auxiliar = "gpt-4.1-mini"  #@param {type:"string"}
modelo_embeddings = "text-embedding-3-small"  #@param {type:"string"}
k_documentos = 12  #@param {type:"integer"}

# modelo_respuesta: redacta las respuestas.
# modelo_auxiliar: solo traduce cada pregunta al inglés para buscar mejor (los resúmenes están en inglés).
# k_documentos: cuántos artículos se entregan al modelo por pregunta.
client = OpenAI(api_key=clave_api)

# 2. Subir las exportaciones de Web of Science desde tu equipo

Al ejecutar la celda siguiente aparecerá un botón **Elegir archivos**. Puedes seleccionar **varios archivos a la vez** (por ejemplo, todos los `.xls` de la primera búsqueda, aunque cada uno tenga solo 50 artículos).

Primero se piden los archivos de la búsqueda 1 y luego los de la búsqueda 2. Si solo tienes una búsqueda, desmarca `usar_busqueda_2`.

In [ ]:
#@title <b><font color="blue" size="+2"> Nombres de las búsquedas
nombre_busqueda_1 = "IA generativa en investigación educativa"  #@param {type:"string"}
usar_busqueda_2 = True  #@param {type:"boolean"}
nombre_busqueda_2 = "Agentes y automatización de la investigación"  #@param {type:"string"}
carpeta_salida = "/content/resultados"  #@param {type:"string"}

os.makedirs(carpeta_salida, exist_ok=True)

In [ ]:
from google.colab import files
import io

def leer_wos(nombre_archivo, contenido):
    # Lee un archivo exportado de WoS (.xls, .xlsx, .csv o .txt delimitado por tabulaciones)
    nombre = nombre_archivo.lower()
    datos = io.BytesIO(contenido)
    if nombre.endswith(".csv"):
        return pd.read_csv(datos)
    if nombre.endswith(".txt"):
        return pd.read_csv(datos, sep="\t", quoting=3, index_col=False)
    try:
        return pd.read_excel(datos)
    except Exception:
        datos.seek(0)
        return pd.read_excel(datos, engine="xlrd")

def subir_busqueda(nombre_busqueda):
    print(f"\nSelecciona TODOS los archivos de la búsqueda: {nombre_busqueda}")
    subidos = files.upload()
    tablas = []
    for nombre_archivo, contenido in subidos.items():
        t = leer_wos(nombre_archivo, contenido)
        t["busqueda"] = nombre_busqueda
        t["archivo_origen"] = nombre_archivo
        tablas.append(t)
        print(f"  {nombre_archivo}: {len(t)} registros")
    if not tablas:
        print("  No se subieron archivos para esta búsqueda.")
    return tablas

partes = subir_busqueda(nombre_busqueda_1)
if usar_busqueda_2:
    partes += subir_busqueda(nombre_busqueda_2)

if not partes:
    raise ValueError("No se cargó ningún archivo. Vuelve a ejecutar la celda y selecciona los archivos.")

df = pd.concat(partes, ignore_index=True)
print(f"\nTotal de registros cargados: {len(df)}")

# Verificar columnas esperadas en la exportación de WoS
for col in ["Article Title", "Abstract"]:
    if col not in df.columns:
        raise ValueError(f"Falta la columna '{col}'. Columnas disponibles: {list(df.columns)}")

# Clave única: ID de WoS, si no DOI, si no el título
def clave_registro(fila):
    for col in ["UT (Unique WOS ID)", "DOI"]:
        if col in fila and pd.notna(fila[col]) and str(fila[col]).strip():
            return str(fila[col]).strip().lower()
    return str(fila["Article Title"]).strip().lower()

df["clave"] = df.apply(clave_registro, axis=1)

# Si un artículo aparece en ambas búsquedas (o en dos archivos), se conserva una vez con todas sus etiquetas
df["busqueda"] = df.groupby("clave")["busqueda"].transform(lambda s: " + ".join(sorted(set(s))))
n_antes = len(df)
df = df.drop_duplicates("clave").reset_index(drop=True)
print(f"Duplicados eliminados: {n_antes - len(df)}")

# Excluir registros sin resumen
sin_resumen = df["Abstract"].isna() | (df["Abstract"].astype(str).str.strip().str.len() < 50)
print(f"Registros sin resumen (excluidos): {sin_resumen.sum()}")
df = df[~sin_resumen].reset_index(drop=True)
print(f"Total de artículos para el análisis: {len(df)}")
print(df["busqueda"].value_counts())


Selecciona TODOS los archivos de la búsqueda: IA generativa en investigación educativa


Saving savedrecs (11).xls to savedrecs (11) (2).xls
Saving savedrecs (12).xls to savedrecs (12) (2).xls
Saving savedrecs (13).xls to savedrecs (13) (2).xls
Saving savedrecs (14).xls to savedrecs (14) (2).xls
Saving savedrecs (15).xls to savedrecs (15) (2).xls
Saving savedrecs (16).xls to savedrecs (16) (2).xls
Saving savedrecs (17).xls to savedrecs (17) (2).xls
Saving savedrecs (18).xls to savedrecs (18) (2).xls
Saving savedrecs (19).xls to savedrecs (19) (2).xls
Saving savedrecs (20).xls to savedrecs (20) (2).xls
  savedrecs (11) (2).xls: 50 registros
  savedrecs (12) (2).xls: 50 registros
  savedrecs (13) (2).xls: 50 registros
  savedrecs (14) (2).xls: 50 registros
  savedrecs (15) (2).xls: 50 registros
  savedrecs (16) (2).xls: 50 registros
  savedrecs (17) (2).xls: 50 registros
  savedrecs (18) (2).xls: 50 registros
  savedrecs (19) (2).xls: 50 registros
  savedrecs (20) (2).xls: 42 registros

Selecciona TODOS los archivos de la búsqueda: Agentes y automatización de la investigació

Saving savedrecs (21).xls to savedrecs (21) (2).xls
  savedrecs (21) (2).xls: 19 registros

Total de registros cargados: 511
Duplicados eliminados: 3
Registros sin resumen (excluidos): 4
Total de artículos para el análisis: 504
busqueda
IA generativa en investigación educativa                                                   485
Agentes y automatización de la investigación                                                16
Agentes y automatización de la investigación + IA generativa en investigación educativa      3
Name: count, dtype: int64


# 3. Preparar el corpus

Cada artículo se convierte en un texto (título + resumen) con una cita corta y una referencia completa. Las referencias las arma el código a partir de los datos de WoS, **no el modelo**, para evitar referencias inventadas.

In [ ]:
def valor(fila, col):
    v = fila.get(col, "")
    return "" if pd.isna(v) else str(v).strip()

def anio(fila):
    for col in ["Publication Year", "Early Access Date"]:
        v = valor(fila, col)
        m = re.search(r"(19|20)\d{2}", v)
        if m:
            return m.group(0)
    return "s.f."

def cita_corta(fila):
    autores = [a.strip() for a in valor(fila, "Authors").split(";") if a.strip()]
    if not autores:
        return f"Sin autor ({anio(fila)})"
    apellido = autores[0].split(",")[0].strip().title()
    sufijo = " et al." if len(autores) > 2 else (f" y {autores[1].split(',')[0].strip().title()}" if len(autores) == 2 else "")
    return f"{apellido}{sufijo} ({anio(fila)})"

def referencia(fila):
    ref = f"{valor(fila, 'Authors')} ({anio(fila)}). {valor(fila, 'Article Title')}. {valor(fila, 'Source Title').title()}."
    doi = valor(fila, "DOI")
    return ref + (f" https://doi.org/{doi}" if doi else "")

corpus = pd.DataFrame({
    "cita": df.apply(cita_corta, axis=1),
    "anio": df.apply(anio, axis=1),
    "titulo": df["Article Title"].fillna("").astype(str),
    "resumen": df["Abstract"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True),
    "doi": df["DOI"].fillna("").astype(str) if "DOI" in df.columns else "",
    "busqueda": df["busqueda"],
    "referencia": df.apply(referencia, axis=1),
})
corpus["texto"] = (corpus["titulo"] + ". " + corpus["resumen"]).str[:8000]

print(corpus["anio"].value_counts().sort_index())
corpus.head()

anio
2022      1
2023     36
2024     73
2025    161
2026    231
2027      2
Name: count, dtype: int64


,cita,anio,titulo,resumen,doi,busqueda,referencia,texto
0,Couture et al. (2026),2026,Sex Education in the Age of Artificial Intelli...,"Large language model chatbots, such as ChatGPT...",10.1007/s10508-026-03575-2,IA generativa en investigación educativa,"Couture, S; Paquette, MM; Guyon, R; Hébert, M;...",Sex Education in the Age of Artificial Intelli...
1,Du y Zhang (2026),2026,AI-assisted IVF company websites: a content an...,STUDY QUESTION How do providers of AI-assisted...,10.1093/hropen/hoag086,IA generativa en investigación educativa,"Du, L; Zhang, YN (2026). AI-assisted IVF compa...",AI-assisted IVF company websites: a content an...
2,García-Fernández et al. (2026),2026,Challenges and opportunities in higher educati...,Purpose This paper explores the effects of art...,10.1108/SBM-01-2026-0020,IA generativa en investigación educativa,"García-Fernández, J; Sevilmis, A; Angosto, S; ...",Challenges and opportunities in higher educati...
3,Zor y Ari (2027),2027,Student perceptions of generative AI in accoun...,Generative AI has outpaced pedagogical and eth...,10.1016/j.ijme.2026.101563,IA generativa en investigación educativa,"Zor, Ü; Ari, O (2027). Student perceptions of ...",Student perceptions of generative AI in accoun...
4,Ülkü et al. (2026),2026,Comparative evaluation of ChatGPT and Gemini f...,OBJECTIVE: Artificial intelligence tools are i...,10.1590/1806-9282.20260869,IA generativa en investigación educativa,"Ülkü, HH; Torun, F; Öncü, S (2026). Comparativ...",Comparative evaluation of ChatGPT and Gemini f...


# 4. Embeddings y búsqueda semántica

Con algunos cientos de resúmenes no hace falta un índice aproximado (Annoy): se calcula la similitud exacta con NumPy, que es más simple y reproducible. El costo de crear los embeddings es muy bajo (centavos de dólar).

In [ ]:
def obtener_embeddings(textos, modelo=modelo_embeddings, lote=100):
    vectores = []
    for i in range(0, len(textos), lote):
        resp = client.embeddings.create(input=textos[i:i+lote], model=modelo)
        vectores.extend([d.embedding for d in resp.data])
        time.sleep(0.5)
    M = np.array(vectores, dtype=np.float32)
    return M / np.linalg.norm(M, axis=1, keepdims=True)

matriz = obtener_embeddings(corpus["texto"].tolist())
print("Embeddings creados:", matriz.shape)

def buscar(consulta, k=k_documentos):
    q = obtener_embeddings([consulta])[0]
    similitudes = matriz @ q
    idx = np.argsort(-similitudes)[:k]
    return corpus.iloc[idx].assign(similitud=similitudes[idx]).reset_index(drop=True)

Embeddings creados: (504, 1536)


# 5. Preguntas

Diez preguntas pensadas para académicos y estudiantes de posgrado. Puedes editarlas o dejar alguna vacía para omitirla.

In [ ]:
#@title <b><font color="blue" size="+2"> Preguntas para la síntesis
pregunta1 = "¿En qué etapas del proceso de investigación (búsqueda y cribado de literatura, extracción de datos, análisis cualitativo, redacción) se está usando la IA generativa y en cuáles muestra resultados más sólidos?"  #@param {type:"string"}
pregunta2 = "¿Qué tan confiable es la IA generativa para codificar datos cualitativos y apoyar el análisis temático en comparación con codificadores humanos, y en qué tipo de tareas o datos falla más?"  #@param {type:"string"}
pregunta3 = "¿Qué evidencia existe sobre la precisión y el ahorro de tiempo al usar modelos de lenguaje en el cribado de estudios y la extracción de datos para revisiones sistemáticas o de alcance?"  #@param {type:"string"}
pregunta4 = "¿Qué estrategias de formulación de prompts (uso de libros de códigos, ejemplos, instrucciones paso a paso, asignación de roles) mejoran la precisión, la consistencia y la trazabilidad de los resultados en tareas de investigación?"  #@param {type:"string"}
pregunta5 = "¿Qué riesgos de alucinaciones, referencias inventadas y sesgos se han documentado al usar IA generativa en investigación, y qué estrategias se proponen para detectarlos y mitigarlos?"  #@param {type:"string"}
pregunta6 = "¿Qué recomendaciones se proponen para proteger datos sensibles, anonimizar información y cumplir principios éticos (consentimiento informado, comités de ética) al procesar datos de investigación con IA generativa?"  #@param {type:"string"}
pregunta7 = "¿Qué modelos de colaboración entre humanos e IA (human-in-the-loop) y qué procedimientos de validación se recomiendan para mantener el rigor metodológico y la integridad científica?"  #@param {type:"string"}
pregunta8 = "¿Qué se recomienda sobre la transparencia y el reporte del uso de IA generativa en publicaciones científicas (declaración de uso, registro de prompts y versiones de modelos, reproducibilidad)?"  #@param {type:"string"}
pregunta9 = "¿Qué beneficios y riesgos se reportan sobre los agentes de IA y las herramientas automatizadas de investigación (deep research, sistemas multiagente, flujos agénticos), y qué tan cerca están de realizar investigación autónoma de calidad?"  #@param {type:"string"}
pregunta10 = "¿Qué implicancias tiene la IA generativa para la formación de investigadores y estudiantes de posgrado: qué competencias se vuelven necesarias y qué riesgos existen de dependencia, pérdida de habilidades o de agencia?"  #@param {type:"string"}

preguntas = [p.strip() for p in [pregunta1, pregunta2, pregunta3, pregunta4, pregunta5,
                                 pregunta6, pregunta7, pregunta8, pregunta9, pregunta10] if p.strip()]
print("Número de preguntas:", len(preguntas))

Número de preguntas: 10


# 6. Instrucciones (prompt) y funciones

El prompt exige responder solo con los documentos entregados, citar con números, distinguir evidencia empírica de propuestas conceptuales y reconocer cuando la información no alcanza.

In [ ]:
def query_llm(prompt, model=modelo_respuesta):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

def consulta_en_ingles(pregunta):
    """Traduce la pregunta al inglés para mejorar la búsqueda en resúmenes escritos en inglés."""
    prompt = ("Translate the following research question into English as a concise academic search query. "
              "Return only the translation.\n\n" + pregunta)
    return query_llm(prompt, model=modelo_auxiliar).strip()

PROMPT = """Eres un investigador experto en metodología de la investigación educativa y en el uso responsable de la inteligencia artificial.
Tu respuesta se presentará en un webinar para académicos y estudiantes de posgrado.

A continuación tienes {n} documentos: títulos y resúmenes de artículos científicos recientes indexados en Web of Science. Cada documento tiene un número entre corchetes.

{contexto}

INSTRUCCIONES
1. Responde la pregunta usando ÚNICAMENTE la información de estos documentos. No agregues conocimiento externo.
2. Cita cada afirmación con el número del documento entre corchetes, por ejemplo [3] o [2, 7]. Usa solo números de la lista; nunca inventes autores, años ni referencias.
3. Distingue entre evidencia empírica (estudios con datos) y propuestas conceptuales, revisiones u opiniones. Señala cuando un hallazgo proviene de una disciplina distinta de la educación.
4. Si los documentos no permiten responder bien algún aspecto de la pregunta, dilo explícitamente.
5. Escribe en español, con tono académico pero claro y directo. No incluyas lista de referencias: se agregará automáticamente.

ESTRUCTURA DE LA RESPUESTA
**Síntesis**: 2 a 3 oraciones que respondan directamente la pregunta.
**Hallazgos clave**: entre 3 y 6 viñetas, cada una con su cita.
**Implicancias para investigadores y estudiantes de posgrado**: 2 a 4 recomendaciones prácticas.
**Advertencias y limitaciones**: riesgos, sesgos o límites de la evidencia disponible.
**Vacíos de investigación**: 1 a 3 preguntas abiertas que la literatura aún no resuelve (útiles para tesis o proyectos).

PREGUNTA
{pregunta}
"""

def armar_contexto(docs):
    bloques = []
    for i, d in docs.iterrows():
        bloques.append(f"[{i+1}] {d['cita']}. {d['titulo']}\nResumen: {d['resumen']}")
    return "\n\n".join(bloques)

def verificar_citas(respuesta, n):
    """Devuelve los números citados válidos y los inválidos (documentos que no existen en el contexto)."""
    numeros = set()
    for grupo in re.findall(r"\[([\d,\s;–-]+)\]", respuesta):
        numeros.update(int(x) for x in re.findall(r"\d+", grupo))
    validos = sorted(x for x in numeros if 1 <= x <= n)
    invalidos = sorted(x for x in numeros if not 1 <= x <= n)
    return validos, invalidos

# 7. Responder las preguntas

In [ ]:
resultados = []
recuperados = []

for num, pregunta in enumerate(preguntas, start=1):
    print("=" * 60)
    print(f"PREGUNTA {num}: {pregunta}")

    consulta = consulta_en_ingles(pregunta)
    docs = buscar(consulta)
    prompt = PROMPT.format(n=len(docs), contexto=armar_contexto(docs), pregunta=pregunta)
    respuesta = query_llm(prompt)

    validos, invalidos = verificar_citas(respuesta, len(docs))
    referencias = "\n".join(f"[{i}] {docs.loc[i-1, 'referencia']}" for i in validos)

    if invalidos:
        print(f"ATENCIÓN: el modelo citó documentos inexistentes {invalidos}. Revisa esta respuesta.")
    print(f"Documentos citados: {len(validos)} de {len(docs)}")
    print(respuesta[:600], "...\n")

    resultados.append({
        "n": num,
        "pregunta": pregunta,
        "consulta_en": consulta,
        "respuesta": respuesta,
        "referencias": referencias,
        "docs_citados": len(validos),
        "citas_invalidas": ", ".join(map(str, invalidos)),
        "validacion_humana": "",   # para completar: Sí / Parcial / No
        "comentarios_revisor": "",
    })

    for i, d in docs.iterrows():
        recuperados.append({
            "pregunta_n": num,
            "numero_en_respuesta": i + 1,
            "similitud": round(float(d["similitud"]), 3),
            "citado": "Sí" if (i + 1) in validos else "No",
            "cita": d["cita"],
            "titulo": d["titulo"],
            "doi": d["doi"],
            "busqueda": d["busqueda"],
        })

    time.sleep(2)  # pausa para no saturar la API

PREGUNTA 1: ¿En qué etapas del proceso de investigación (búsqueda y cribado de literatura, extracción de datos, análisis cualitativo, redacción) se está usando la IA generativa y en cuáles muestra resultados más sólidos?
Documentos citados: 11 de 12
**Síntesis**: La IA generativa se está usando en todas las etapas mencionadas: búsqueda y cribado de literatura, extracción y organización de información, análisis cualitativo y redacción; sin embargo, su uso más extendido parece ser la redacción, no necesariamente su uso más eficaz [4, 8, 11]. La evidencia disponible no permite afirmar que la IA supere de forma consistente a los métodos humanos en alguna etapa: hay resultados empíricos útiles sobre búsquedas y apoyo al análisis, pero también errores importantes y una necesidad reiterada de verificación humana [2, 5, 6].

**Hallazgos clave**

 ...

PREGUNTA 2: ¿Qué tan confiable es la IA generativa para codificar datos cualitativos y apoyar el análisis temático en comparación con codificado

# 8. Exportar a Excel (para la validación humana)

El archivo tiene tres hojas:
* **respuestas**: preguntas, respuestas y referencias, con columnas vacías para que un revisor marque si cada respuesta está respaldada.
* **documentos_recuperados**: qué artículos recibió el modelo en cada pregunta y cuáles citó.
* **metodo**: datos para reportar el procedimiento.

In [ ]:
hoy = date.today().strftime("%Y_%m_%d")
informe = pd.DataFrame(resultados)
docs_recuperados = pd.DataFrame(recuperados)
metodo = pd.DataFrame({
    "parametro": ["Fecha de ejecución", "Búsquedas", "Artículos analizados", "Modelo de respuesta",
                  "Modelo auxiliar (traducción de consultas)", "Modelo de embeddings", "Documentos por pregunta"],
    "valor": [date.today().isoformat(),
              nombre_busqueda_1 + (" | " + nombre_busqueda_2 if usar_busqueda_2 else ""),
              len(corpus), modelo_respuesta, modelo_auxiliar, modelo_embeddings, k_documentos],
})

ruta_excel = os.path.join(carpeta_salida, f"respuestas_literatura_IA_{hoy}.xlsx")
with pd.ExcelWriter(ruta_excel) as writer:
    informe.to_excel(writer, sheet_name="respuestas", index=False)
    docs_recuperados.to_excel(writer, sheet_name="documentos_recuperados", index=False)
    metodo.to_excel(writer, sheet_name="metodo", index=False)
print("Excel guardado en:", ruta_excel)
files.download(ruta_excel)

Excel guardado en: /content/resultados/respuestas_literatura_IA_2026_10_27.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Exportar a Word

Las respuestas ya están en español, así que no se necesita una traducción adicional.

In [ ]:
def agregar_texto_con_negritas(parrafo, texto):
    for parte in re.split(r"(\*\*[^*]+\*\*)", texto):
        if parte.startswith("**") and parte.endswith("**"):
            parrafo.add_run(parte[2:-2]).bold = True
        elif parte:
            parrafo.add_run(parte)

def agregar_respuesta(documento, respuesta):
    for linea in respuesta.splitlines():
        linea = linea.strip()
        if not linea or linea.startswith("#") and not linea.strip("#").strip():
            continue
        linea = linea.lstrip("#").strip() if linea.startswith("#") else linea
        if re.match(r"^[-*•]\s+", linea):
            p = documento.add_paragraph(style="List Bullet")
            agregar_texto_con_negritas(p, re.sub(r"^[-*•]\s+", "", linea))
        else:
            agregar_texto_con_negritas(documento.add_paragraph(), linea)

documento = Document()
documento.add_heading("IA generativa en la investigación educativa: síntesis de la literatura reciente", 0)
documento.add_paragraph(
    f"Síntesis generada el {date.today().strftime('%d-%m-%Y')} a partir de {len(corpus)} títulos y resúmenes "
    f"exportados desde Web of Science. Para cada pregunta se recuperaron los {k_documentos} artículos más relacionados "
    f"mediante búsqueda semántica ({modelo_embeddings}) y la respuesta fue redactada por {modelo_respuesta} usando solo esos documentos. "
    "Las referencias se generaron automáticamente desde los datos de WoS. "
    "Este texto fue producido con IA y requiere validación humana antes de citarse o difundirse."
)

for r in resultados:
    documento.add_heading(f"Pregunta {r['n']}", level=1)
    p = documento.add_paragraph()
    p.add_run(r["pregunta"]).italic = True
    agregar_respuesta(documento, r["respuesta"])
    if r["citas_invalidas"]:
        documento.add_paragraph(f"⚠ El modelo citó documentos inexistentes ({r['citas_invalidas']}); revisar.")
    documento.add_heading("Referencias", level=2)
    for ref in r["referencias"].splitlines():
        documento.add_paragraph(ref)

ruta_word = os.path.join(carpeta_salida, f"sintesis_literatura_IA_{hoy}.docx")
documento.save(ruta_word)
print("Word guardado en:", ruta_word)
files.download(ruta_word)

Word guardado en: /content/resultados/sintesis_literatura_IA_2026_10_27.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 10. Validación humana (recomendada)

Antes de usar las respuestas en el webinar:

1. Abre el Excel y, para cada pregunta, revisa al menos dos o tres afirmaciones contra los resúmenes citados (hoja *documentos_recuperados*).
2. Marca en *validacion_humana* si la respuesta está respaldada (Sí / Parcial / No) y anota observaciones.
3. Ten presente que el modelo solo leyó resúmenes, no los artículos completos: los hallazgos clave conviene confirmarlos en el texto original.